In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
# folder containing: jet80, jet170, jet1000, lep80, lep170, lep1000
file_path = Path("/Users/lydiaamstutz/Downloads/jet1000update.root")
sample_label = "jet1000"
print("Opening:", file_path)
file = uproot.open(file_path)
print("File keys:", file.keys())

In [ ]:
# try the expected tree first
if "reco" in file:
    tree = file["reco"]
else:
    # show what trees exist so i can pick the right one
    print("Top-level keys:", file.keys())
    raise KeyError("Tree 'reco' not found. Pick the correct tree from file.keys().")

print("Tree loaded:", tree)
print("N entries:", tree.num_entries)

In [ ]:
# find what the tau trigger branches are actually called in this ntuple
keys = tree.keys()
tau_like = [k for k in keys if ("trigPassed_HLT_tau" in k) or ("tau50" in k) or ("mediumRNN" in k)]
print("found", len(tau_like), "tau-ish branches")
for k in tau_like[:80]:
    print(k)

In [ ]:
# find tau pt branch name in this ntuple
tau_pt_candidates = [k for k in tree.keys() if ("tau" in k.lower()) and ("pt" in k.lower())]
print("tau pt candidates:", tau_pt_candidates)

In [ ]:
# find jet pt branch name in this ntuple
jet_pt_candidates = [k for k in tree.keys() if ("jet" in k.lower()) and ("pt" in k.lower())]
print("jet pt candidates:", jet_pt_candidates)

In [ ]:
##check that the file loaded correctly
branches = [
    "eventNumber",
    "met_met_NOSYS",
    "trigPassed_HLT_xe65_cell_xe90_pfopufit_L1XE50",
    "trigPassed_HLT_xe80_cell_xe115_tcpufit_L1XE50",
    "trigPassed_HLT_j70_0eta290_020jvt_bgn160_3j70_pf_ftf_preselj50b85XX3j50_L14J20",
    "trigPassed_HLT_j75_0eta290_020jvt_bgn160_3j75_pf_ftf_preselj50b85XX3j50_L14J20",
    "tau_pt_NOSYS",
    "jet_pt_NOSYS",
    "jet_GN2v01_FixedCutBEff_85_select",
]

#add the taus
branches += [
    "trigPassed_HLT_tau50_mediumRNN_tracktwoMVA_xe80_tcpufit_xe50_cell_L1XE50",
    "trigPassed_HLT_tau50_mediumRNN_tracktwoMVA_xe80_pfopufit_xe50_cell_L1XE50",
    "trigPassed_HLT_tau160_mediumRNN_tracktwoMVA_L1TAU100",
]

branches += [
    "trigPassed_HLT_2j35_0eta290_020jvt_bgn160_3j35_pf_ftf_presel3j25XX2j25b85_L15J15p0ETA25",   # J_2J35
    "trigPassed_HLT_2j45_0eta290_020jvt_bgn160_2j45_pf_ftf_presel2j25XX2j25b85_L14J15p0ETA25",   # J_2J45
    "trigPassed_HLT_j140_2j50_0eta290_020jvt_bgn170_pf_ftf_preselj80XX2j45b90_L1J85_3J30",   # J140_2J50
    "trigPassed_HLT_j150_2j55_0eta290_020jvt_bgn170_pf_ftf_preselj80XX2j45b90_L1J85_3J30",   # J150_2J55
    "trigPassed_HLT_j175_0eta290_020jvt_bgn160_j60_0eta290_020jvt_bgn160_pf_ftf_preselj140b85XXj45b85_L1J100",   # J175_J60
    "trigPassed_HLT_tau35_mediumRNN_tracktwoMVA_tau25_mediumRNN_tracktwoMVA_03dRAB_L1TAU20IM_2TAU12IM_4J12p0ETA25",   # DITAU_35_25
    "trigPassed_HLT_tau35_mediumRNN_tracktwoMVA_tau25_mediumRNN_tracktwoMVA_03dRAB30_L1DR_TAU20ITAU12I_J25",   # DITAU_35_25_J
    "trigPassed_HLT_tau40_mediumRNN_tracktwoMVA_tau35_mediumRNN_tracktwoMVA_03dRAB_L1TAU25IM_2TAU20IM_2J25_3J20",   # DITAU_40_35
    "trigPassed_HLT_tau80_mediumRNN_tracktwoMVA_tau35_mediumRNN_tracktwoMVA_03dRAB30_L1TAU60_DR_TAU20ITAU12I",   # DITAU_80_35
    "trigPassed_HLT_tau80_mediumRNN_tracktwoMVA_tau60_mediumRNN_tracktwoMVA_03dRAB_L1TAU60_2TAU40",   # DITAU_80_60
]

arr = tree.arrays(branches, library="ak")
data = {}
for k in branches:
    if arr[k].ndim == 1:
        data[k] = np.array(arr[k])
    else:
        data[k] = arr[k].to_list()

df = pd.DataFrame(data)

offline_met_gev = df["met_met_NOSYS"].values / 1000.0

In [ ]:
print("rows:", len(df))
print("unique events:", df["eventNumber"].nunique())
print("any NaNs in MET:", df["met_met_NOSYS"].isna().any())
print("MET min/max:", df["met_met_NOSYS"].min(), df["met_met_NOSYS"].max())

In [ ]:
met_pf = (df["trigPassed_HLT_xe65_cell_xe90_pfopufit_L1XE50"] == 1)
met_hi = (df["trigPassed_HLT_xe80_cell_xe115_tcpufit_L1XE50"] == 1)
j70 = (df["trigPassed_HLT_j70_0eta290_020jvt_bgn160_3j70_pf_ftf_preselj50b85XX3j50_L14J20"] == 1)
j75 = (df["trigPassed_HLT_j75_0eta290_020jvt_bgn160_3j75_pf_ftf_preselj50b85XX3j50_L14J20"] == 1)
tau_tcp = (df["trigPassed_HLT_tau50_mediumRNN_tracktwoMVA_xe80_tcpufit_xe50_cell_L1XE50"] == 1)
tau_pf = (df["trigPassed_HLT_tau50_mediumRNN_tracktwoMVA_xe80_pfopufit_xe50_cell_L1XE50"] == 1)
tau160 = (df["trigPassed_HLT_tau160_mediumRNN_tracktwoMVA_L1TAU100"] == 1)

# new triggers from the *update.root file
j_2j35 = (df["trigPassed_HLT_2j35_0eta290_020jvt_bgn160_3j35_pf_ftf_presel3j25XX2j25b85_L15J15p0ETA25"] == 1)
j_2j45 = (df["trigPassed_HLT_2j45_0eta290_020jvt_bgn160_2j45_pf_ftf_presel2j25XX2j25b85_L14J15p0ETA25"] == 1)
j140_2j50 = (df["trigPassed_HLT_j140_2j50_0eta290_020jvt_bgn170_pf_ftf_preselj80XX2j45b90_L1J85_3J30"] == 1)
j150_2j55 = (df["trigPassed_HLT_j150_2j55_0eta290_020jvt_bgn170_pf_ftf_preselj80XX2j45b90_L1J85_3J30"] == 1)
j175_j60 = (df["trigPassed_HLT_j175_0eta290_020jvt_bgn160_j60_0eta290_020jvt_bgn160_pf_ftf_preselj140b85XXj45b85_L1J100"] == 1)
ditau_35_25 = (df["trigPassed_HLT_tau35_mediumRNN_tracktwoMVA_tau25_mediumRNN_tracktwoMVA_03dRAB_L1TAU20IM_2TAU12IM_4J12p0ETA25"] == 1)
ditau_35_25_j = (df["trigPassed_HLT_tau35_mediumRNN_tracktwoMVA_tau25_mediumRNN_tracktwoMVA_03dRAB30_L1DR_TAU20ITAU12I_J25"] == 1)
ditau_40_35 = (df["trigPassed_HLT_tau40_mediumRNN_tracktwoMVA_tau35_mediumRNN_tracktwoMVA_03dRAB_L1TAU25IM_2TAU20IM_2J25_3J20"] == 1)
ditau_80_35 = (df["trigPassed_HLT_tau80_mediumRNN_tracktwoMVA_tau35_mediumRNN_tracktwoMVA_03dRAB30_L1TAU60_DR_TAU20ITAU12I"] == 1)
ditau_80_60 = (df["trigPassed_HLT_tau80_mediumRNN_tracktwoMVA_tau60_mediumRNN_tracktwoMVA_03dRAB_L1TAU60_2TAU40"] == 1)

In [ ]:
from itertools import combinations

base = {
    "PF_MET":      met_pf,
    "TCP_MET":     met_hi,
    "J70":         j70,
    "J75":         j75,
    "TAU+MET_PF":  tau_pf,
    "TAU+MET_TCP": tau_tcp,
    "TAU160":      tau160,
    # new triggers
    "J_2J35": j_2j35,
    "J_2J45": j_2j45,
    "J140_2J50": j140_2j50,
    "J150_2J55": j150_2j55,
    "J175_J60": j175_j60,
    "DITAU_35_25": ditau_35_25,
    "DITAU_35_25_J": ditau_35_25_j,
    "DITAU_40_35": ditau_40_35,
    "DITAU_80_35": ditau_80_35,
    "DITAU_80_60": ditau_80_60,
}

# Sanity check: any banch thatfires on 0 events is likely unfilled in
# this MC production, not real 0% efficiency.
zero_fire = [name for name, m in base.items() if m.sum() == 0]
if zero_fire:
    print("[WARNING] These trigger branches fire on 0 events -- likely unfilled:")
    for n in zero_fire:
        print(f"   - {n}")
    print()

# build all or/and combos
OR = {}
AND = {}
for a, b in combinations(base.keys(), 2):
    OR[f"{a} OR {b}"] = base[a] | base[b]
    AND[f"{a} AND {b}"] = base[a] & base[b]

print(f"Built {len(OR)} OR combos and {len(AND)} AND combos.")

# Overall efficiencies: triggers
print("\n=== Overall efficiencies (Triggers) ===")
for name in base:
    print(f"{name:15s}  eff={base[name].mean():.4f}  passed={base[name].sum()}")

# Overall efficiencies: all ORs
print("\n=== Overall efficiencies (All OR combos) ===")
for name in sorted(OR.keys()):
    print(f"{name:55s} eff={OR[name].mean():.4f}")

# Overall efficiencies: all ANDs
print("\n=== Overall efficiencies (All AND combos) ===")
for name in sorted(AND.keys()):
    print(f"{name:55s} eff={AND[name].mean():.4f}")

# A few specific combos
print("\n=== Specific combos ===")
print("PF_MET OR TAU+MET_PF:", OR["PF_MET OR TAU+MET_PF"].mean())
print("J70 OR TAU+MET_PF:",    OR["J70 OR TAU+MET_PF"].mean())
print("PF_MET OR J70:",        OR["PF_MET OR J70"].mean())

In [ ]:
from itertools import combinations

print("\n=== Overlap counts (A only / B only / both / neither) ===")
for a, b in combinations(base.keys(), 2):
    A = base[a]
    B = base[b]
    a_only = (A & ~B).sum()
    b_only = (~A & B).sum()
    both   = (A & B).sum()
    neither= (~A & ~B).sum()

    # usefl fractions (avoid divide-by-zero)
    frac_a_not_b = (a_only / A.sum()) if A.sum() > 0 else np.nan
    frac_b_not_a = (b_only / B.sum()) if B.sum() > 0 else np.nan

    print(f"{a:14s} vs {b:14s} | "
          f"Aonly={a_only:7d} Bonly={b_only:7d} both={both:7d} neither={neither:7d} | "
          f"frac(A not B)={frac_a_not_b:.3f}  frac(B not A)={frac_b_not_a:.3f}")

In [ ]:
# binomial efficiency + 1-sigma uncertainty ----
def eff_err(passed, total):
    if total == 0:
        return np.nan, np.nan
    eff = passed / total
    err = np.sqrt(eff * (1 - eff) / total)
    return eff, err

bins = np.linspace(0, 350, 30)
centers = 0.5 * (bins[:-1] + bins[1:])

# choose a focused set to plot + quantify
focus = {
    "PF_MET": base["PF_MET"],
    "TCP_MET": base["TCP_MET"],
    "J70": base["J70"],
    "TAU+MET_PF": base["TAU+MET_PF"],
    "TAU+MET_TCP": base["TAU+MET_TCP"],
    "PF_MET OR J70": OR["PF_MET OR J70"],
    "PF_MET OR TAU+MET_PF": OR["PF_MET OR TAU+MET_PF"],
    "J70 OR TAU+MET_PF": OR["J70 OR TAU+MET_PF"],
    "PF_MET AND TAU+MET_PF": AND["PF_MET AND TAU+MET_PF"],
    "TAU160": base["TAU160"],
    "PF_MET OR TAU160": OR["PF_MET OR TAU160"],
    "J70 OR TAU160": OR["J70 OR TAU160"],
    # new triggers worth comparing on the turn-on plot
    "J_2J45": base["J_2J45"],
    "PF_MET OR J_2J45": OR["PF_MET OR J_2J45"],
    "DITAU_35_25": base["DITAU_35_25"],
    "PF_MET OR DITAU_35_25": OR["PF_MET OR DITAU_35_25"],
}

# storage
Nbin = []
out = {k: {"eff": [], "err": []} for k in focus.keys()}

for i in range(len(bins) - 1):
    lo, hi = bins[i], bins[i+1]
    sel = (offline_met_gev >= lo) & (offline_met_gev < hi)
    total = sel.sum()
    Nbin.append(total)

    for name, mask in focus.items():
        if total < 50:
            out[name]["eff"].append(np.nan)
            out[name]["err"].append(np.nan)
        else:
            passed = (mask & sel).sum()
            e, s = eff_err(passed, total)
            out[name]["eff"].append(e)
            out[name]["err"].append(s)

# plot with error bars
plt.figure(figsize=(10, 6))
for name in ["PF_MET", "TCP_MET", "J70", "TAU+MET_PF", "TAU+MET_TCP",
             "TAU160",
             "PF_MET OR J70", "PF_MET OR TAU+MET_PF", "J70 OR TAU+MET_PF",
             "PF_MET OR TAU160", "J70 OR TAU160",
             "J_2J45", "PF_MET OR J_2J45",
             "DITAU_35_25", "PF_MET OR DITAU_35_25"]:
    plt.errorbar(
        centers,
        out[name]["eff"],
        yerr=out[name]["err"],
        fmt="o",
        label=name
    )

plt.xlabel("Offline MET [GeV]")
plt.ylabel("Efficiency")
plt.ylim(0, 1.05)
plt.legend(fontsize=8, ncol=2)
plt.title(f"Trigger turn-on vs offline MET — {sample_label}")
plt.show()

# quantified table: 100–200 GeV
print("\n=== Per-bin quantification (100–200 GeV offline MET) ===")
print("bin[GeV]      N    PF_MET        TAU+MET_PF    PF_MET OR TAU+MET_PF  J70 OR TAU+MET_PF  PF_MET OR J_2J45  PF_MET OR DITAU_35_25")
print("-"*150)
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    if lo < 100 or hi > 200:
        continue
    total = Nbin[i]
    if total < 50:
        continue

    def fmt(name):
        e = out[name]["eff"][i]
        s = out[name]["err"][i]
        return f"{e:6.3f}±{s:5.3f}"

    print(f"[{lo:5.1f},{hi:5.1f})  {total:5d}  "
          f"{fmt('PF_MET'):>12s}  "
          f"{fmt('TAU+MET_PF'):>12s}  "
          f"{fmt('PF_MET OR TAU+MET_PF'):>20s}  "
          f"{fmt('J70 OR TAU+MET_PF'):>17s}  "
          f"{fmt('PF_MET OR J_2J45'):>16s}  "
          f"{fmt('PF_MET OR DITAU_35_25'):>20s}")

In [ ]:
# tau trigger efficiency vs leading tau pt
lead_tau_pt = []
for i in range(len(df)):
    pts = df["tau_pt_NOSYS"][i]   # list of tau pts (MeV)
    if len(pts) > 0:
        lead_tau_pt.append(max(pts) / 1000.0)   # GeV
    else:
        lead_tau_pt.append(np.nan)

lead_tau_pt = np.array(lead_tau_pt)
print("lead tau pt min/max (GeV):", np.nanmin(lead_tau_pt), np.nanmax(lead_tau_pt))

bins_tau = np.linspace(0, 300, 31)
centers = []
eff_tau_pf_pt = []
eff_tau_tcp_pt = []
eff_tau160_pt = []

for i in range(len(bins_tau) - 1):
    lo = bins_tau[i]
    hi = bins_tau[i + 1]
    sel = (lead_tau_pt >= lo) & (lead_tau_pt < hi) & np.isfinite(lead_tau_pt)
    centers.append((lo + hi) / 2)

    if sel.sum() < 20:    # smaller cutoff so don't get a billion NaNs
        eff_tau_pf_pt.append(np.nan)
        eff_tau_tcp_pt.append(np.nan)
        eff_tau160_pt.append(np.nan)
    else:
        eff_tau_pf_pt.append(tau_pf[sel].mean())
        eff_tau_tcp_pt.append(tau_tcp[sel].mean())
        eff_tau160_pt.append(tau160[sel].mean())

plt.figure()
plt.plot(centers, eff_tau_pf_pt, "o", label="tau_pf")
plt.plot(centers, eff_tau_tcp_pt, "o", label="tau_tcp")
plt.plot(centers, eff_tau160_pt, "o", label="tau160")
plt.xlabel("leading tau pT")
plt.legend()
plt.title(f"Tau trigger efficiency vs leading tau pT — {sample_label}")

from pathlib import Path
PLOT_DIR = Path("plots")
PLOT_DIR.mkdir(exist_ok=True)

plt.savefig(PLOT_DIR / "1000tau_pt_turnon_overlay.png",
            dpi=300,
            bbox_inches="tight")
plt.show()

In [ ]:
# b-tag trigger efficiency vs leading b-jet pt (b85 jets)
lead_bjet_pt = []
for i in range(len(df)):
    pts = df["jet_pt_NOSYS"][i]                          # jet pts (MeV)
    b85 = df["jet_GN2v01_FixedCutBEff_85_select"][i]     # 0/1 flags

    bpts = []
    for j in range(len(pts)):
        if b85[j] == 1:
            bpts.append(pts[j])

    if len(bpts) > 0:
        lead_bjet_pt.append(max(bpts) / 1000.0)          # GeV
    else:
        lead_bjet_pt.append(np.nan)

lead_bjet_pt = np.array(lead_bjet_pt)
print("lead b-jet pt min/max (GeV):", np.nanmin(lead_bjet_pt), np.nanmax(lead_bjet_pt))
print("fraction of events with >=1 b85 jet:", np.isfinite(lead_bjet_pt).mean())

bins_b = np.linspace(0, 400, 33)
centers = []
eff_j70_bpt = []
eff_j75_bpt = []

for i in range(len(bins_b) - 1):
    lo = bins_b[i]
    hi = bins_b[i + 1]
    sel = (lead_bjet_pt >= lo) & (lead_bjet_pt < hi) & np.isfinite(lead_bjet_pt)
    centers.append((lo + hi) / 2)

    if sel.sum() < 20:    # smaller cutoff because b-jets might be rarer????
        eff_j70_bpt.append(np.nan)
        eff_j75_bpt.append(np.nan)
    else:
        eff_j70_bpt.append(j70[sel].mean())
        eff_j75_bpt.append(j75[sel].mean())

plt.figure()
plt.plot(centers, eff_j70_bpt, "o", label="J70")
plt.plot(centers, eff_j75_bpt, "o", label="J75")
plt.xlabel("leading b-jet pT [GeV] (b85)")
plt.ylabel("efficiency")
plt.ylim(0, 1.05)
plt.legend()
plt.title("B-tag trigger efficiency vs leading b-jet pT")
plt.show()

In [ ]:
print("Trigger Study Summary")

# basic efficiencies
print("Basic efficiencies:")
print(" PF-MET:", round(met_pf.mean(), 3))
print(" J70:",    round(j70.mean(), 3))
print(" J75:",    round(j75.mean(), 3))
print(" TAU pf:", round(tau_pf.mean(), 3))
print(" TAU tcp:", round(tau_tcp.mean(), 3))
print(" TAU160:",  round(tau160.mean(), 3))
# new triggers
print(" J_2J35:", round(j_2j35.mean(), 3))
print(" J_2J45:", round(j_2j45.mean(), 3))
print(" J140_2J50:", round(j140_2j50.mean(), 3))
print(" J150_2J55:", round(j150_2j55.mean(), 3))
print(" J175_J60:", round(j175_j60.mean(), 3))
print(" DITAU_35_25:", round(ditau_35_25.mean(), 3))
print(" DITAU_35_25_J:", round(ditau_35_25_j.mean(), 3))
print(" DITAU_40_35:", round(ditau_40_35.mean(), 3))
print(" DITAU_80_35:", round(ditau_80_35.mean(), 3))
print(" DITAU_80_60:", round(ditau_80_60.mean(), 3))
print()

# overlaps
print("Overlaps with PF-MET:")
j70_only = (j70 & ~met_pf).sum()
j75_only = (j75 & ~met_pf).sum()

if j70.sum() > 0:
    print(" frac(J70 not PF):", round(j70_only / j70.sum(), 3))
if j75.sum() > 0:
    print(" frac(J75 not PF):", round(j75_only / j75.sum(), 3))

tau_only = (tau_pf & ~met_pf).sum()
if tau_pf.sum() > 0:
    print(" frac(tau_pf not PF):", round(tau_only / tau_pf.sum(), 3))

tau160_only = (tau160 & ~met_pf).sum()
if tau160.sum() > 0:
    print(" frac(tau160 not PF):", round(tau160_only / tau160.sum(), 3))

# new-trigger overlaps with PF-MET
for name, mask in [("J_2J35", j_2j35), ("J_2J45", j_2j45),
                   ("J140_2J50", j140_2j50), ("J150_2J55", j150_2j55),
                   ("J175_J60", j175_j60),
                   ("DITAU_35_25", ditau_35_25),
                   ("DITAU_35_25_J", ditau_35_25_j),
                   ("DITAU_40_35", ditau_40_35),
                   ("DITAU_80_35", ditau_80_35),
                   ("DITAU_80_60", ditau_80_60)]:
    only = (mask & ~met_pf).sum()
    if mask.sum() > 0:
        print(f" frac({name} not PF):", round(only / mask.sum(), 3))

print()

# OR gains
print("OR efficiencies:")
print(" PF only:",                  round(met_pf.mean(), 3))
print(" PF OR J70:",                round((met_pf | j70).mean(), 3))
print(" PF OR J75:",                round((met_pf | j75).mean(), 3))
print(" PF OR J70 OR J75:",         round((met_pf | j70 | j75).mean(), 3))
print(" PF OR J70 OR tau:",         round((met_pf | j70 | tau_pf).mean(), 3))
print(" PF OR all (excl tau160):",  round((met_pf | met_hi | j70 | j75 | tau_pf | tau_tcp).mean(), 3))
print(" PF OR all (incl tau160):",  round((met_pf | met_hi | j70 | j75 | tau_pf | tau_tcp | tau160).mean(), 3))
print(" TAU160 OR PF_MET:",         round((tau160 | met_pf).mean(), 3))
print(" TAU160 OR J70:",            round((tau160 | j70).mean(), 3))
print()

# new-trigger OR gains
print(" PF OR J_2J45:",             round((met_pf | j_2j45).mean(), 3))
print(" PF OR J_2J35:",             round((met_pf | j_2j35).mean(), 3))
print(" PF OR J140_2J50:",          round((met_pf | j140_2j50).mean(), 3))
print(" PF OR DITAU_35_25:",        round((met_pf | ditau_35_25).mean(), 3))
print(" PF OR DITAU_40_35:",        round((met_pf | ditau_40_35).mean(), 3))
print(" PF OR all new b-tag:",      round((met_pf | j_2j35 | j_2j45 | j140_2j50 | j150_2j55 | j175_j60).mean(), 3))
print(" PF OR all new di-tau:",     round((met_pf | ditau_35_25 | ditau_35_25_j | ditau_40_35 | ditau_80_35 | ditau_80_60).mean(), 3))
print(" PF OR all new triggers:",   round((met_pf | j_2j35 | j_2j45 | j140_2j50 | j150_2j55 | j175_j60 |
                                            ditau_35_25 | ditau_35_25_j | ditau_40_35 | ditau_80_35 | ditau_80_60).mean(), 3))
print(" PF OR EVERYTHING:",         round((met_pf | met_hi | j70 | j75 | tau_pf | tau_tcp | tau160 |
                                            j_2j35 | j_2j45 | j140_2j50 | j150_2j55 | j175_j60 |
                                            ditau_35_25 | ditau_35_25_j | ditau_40_35 | ditau_80_35 | ditau_80_60).mean(), 3))

In [ ]:
#  Offline MET cut scan (100→200 GeV, 5 GeV steps) under PF MET trigger 

# Per the meeting: apply PF_MET trigger as the baseline, then sweep the
# offline MET threshold from 100 to 200 GeV in 5-GeV steps to see how
# much efficiency we lose/gain as that cut moves.

cut_values = np.arange(100, 205, 5)   # 100, 105, ..., 200 GeV

# Events that pass the PF MET trigger
pf_mask = met_pf.values    # boolean array

# Total events passing PF trigger (our denominator)
n_pf_total = pf_mask.sum()

effs = []
errors = []

for cut in cut_values:
    passed = (pf_mask & (offline_met_gev >= cut)).sum()
    eff, err = eff_err(passed, n_pf_total)
    effs.append(eff)
    errors.append(err)

effs = np.array(effs)
errors = np.array(errors)

#  Plot 
fig, ax = plt.subplots(figsize=(9, 5))

ax.errorbar(cut_values, effs, yerr=errors, fmt='o-', color='steelblue',
            capsize=3, label='PF MET trigger + offline cut')
ax.axvline(150, color='red', linestyle='--', label='Current cut (150 GeV)')

ax.set_xlabel('Offline MET cut [GeV]')
ax.set_ylabel('Efficiency (events passing trigger AND cut / events passing trigger)')
ax.set_ylim(0, 1.05)
ax.set_title(f'Efficiency vs offline MET cut — {sample_label} (PF MET baseline)')
ax.legend()
plt.tight_layout()
plt.show()

#  Numeric table 
print(f"\n=== Offline MET cut scan — {sample_label} (denominator = PF MET trigger events: {n_pf_total}) ===")
print(f"{'Cut [GeV]':>10}  {'Efficiency':>12}  {'Unc':>8}  {'Rel. to 150 GeV':>16}")
print("-" * 55)

ref_eff = effs[cut_values == 150][0]
for c, e, s in zip(cut_values, effs, errors):
    delta = e - ref_eff
    marker = " ← current" if c == 150 else ""
    print(f"{c:10.0f}  {e:12.4f}  {s:8.4f}  {delta:+.4f}{marker}")